# Airbridge Entry API - 신규 앱 온보딩 노트북

새로운 앱을 Airbridge Entry API에 연동할 때 사용하는 노트북입니다.  
위에서 아래로 순서대로 실행하면 됩니다.

### 이 노트북이 하는 일
1. 앱 정보 입력
2. 이벤트 목록 조회 (Snowflake 또는 CSV)
3. 이벤트 매핑 확인 (기본 커머스 매핑과 비교)
4. 매핑 수정 (필요시)
5. config 저장
6. 데이터 추출 테스트
7. D3 Purchase/Churn 모델 학습
8. 서버에 모델 업로드
9. 고객사 안내사항 출력

### 소요 시간
- 이벤트 매핑이 기본값 그대로 맞으면: ~10분
- 커스텀 매핑 필요하면: ~30분

### 사전 준비
- Airbridge 대시보드에서 앱의 `app_id` 확인
- 서버 URL 확인 (Render 또는 로컬)
- Python 환경 (pandas, scikit-learn, requests)

---
## Step 1: 앱 정보 입력

연동할 앱의 기본 정보를 입력합니다.

- **APP_ID**: Airbridge 대시보드에서 확인할 수 있는 앱 고유 번호 (Settings → App Info)
- **APP_NAME**: 서버에서 모델/config를 구분하는 이름 (영문, 소문자, 공백 없이)
- **SERVER_URL**: Entry API 서버 주소

In [ ]:
APP_ID = None  # ← 여기에 새 앱의 Airbridge app_id를 입력하세요
assert APP_ID is not None, "APP_ID를 입력하세요!"

APP_NAME = "ablog"  # ← 서버에서 사용할 앱 이름
SERVER_URL = "https://airbridge-entry-api-prototype.onrender.com"

print(f"앱 ID: {APP_ID}")
print(f"앱 이름: {APP_NAME}")
print(f"서버: {SERVER_URL}")

### 데이터 폴더 구조

이 노트북에서 `data/{APP_NAME}/` 폴더를 만들고, 이후 모든 데이터는 여기에 쌓입니다:

```
data/{APP_NAME}/
├── weekly_2026-04-03.csv    ← 온보딩 시 첫 데이터 (과거 30~60일)
├── weekly_2026-04-07.csv    ← 매주 1주일치 추가
├── weekly_2026-04-14.csv    ← 최신 파일을 자동으로 사용
└── rct_data.csv             ← Phase 1 RCT 데이터 (추후)
```

노트북은 가장 최신 `weekly_*.csv` 파일을 자동으로 찾아 사용합니다.

---
## Step 2: 이벤트 목록 조회

Snowflake에서 이 앱의 이벤트 종류를 조회합니다.  
어떤 이벤트가 실제로 태깅되어 들어오고 있는지 확인하는 단계입니다.

프로토타입에서는 Snowflake 대신 CSV로 대체합니다.  
실서비스 연동 시에는 주석 처리된 Snowflake 코드를 사용하세요.

In [ ]:
import pandas as pd
import json
import os

# === 방법 1: Snowflake에서 직접 조회 (실서비스) ===
# conn = snowflake.connector.connect(...)
# query = f"""
#     SELECT DATA__EVENTDATA__GOAL__CATEGORY AS goal_category, COUNT(*) AS cnt
#     FROM AIRBRIDGE.PUBLIC.INTERNAL_MOBILE_EVENTS  
#     WHERE DATA__APP__APPID = {APP_ID}
#     AND DATA__EVENTDATA__CATEGORY LIKE '9360%'
#     GROUP BY 1 ORDER BY cnt DESC LIMIT 30
# """
# df_events = pd.read_sql(query, conn)

# === 방법 2: Snowflake에서 CSV로 내보낸 후 로드 ===
# 1. 위 쿼리를 Snowflake Web UI에서 실행
# 2. 결과를 CSV로 다운로드
# 3. query_and_sample/ 폴더에 저장
csv_path = f'../query_and_sample/events_{APP_NAME}.csv'
if os.path.exists(csv_path):
    df_events = pd.read_csv(csv_path)
    print(f"✅ {csv_path} 로드 완료 ({len(df_events)}개 이벤트)")
else:
    print(f"⚠️ {csv_path} 파일이 없습니다!")
    print(f"")
    print(f"Snowflake에서 아래 쿼리를 실행하고 CSV로 저장해주세요:")
    print(f"  파일명: query_and_sample/events_{APP_NAME}.csv")
    print(f"")
    print(f"  SELECT DATA__EVENTDATA__GOAL__CATEGORY AS GOAL_CATEGORY, COUNT(*) AS CNT")
    print(f"  FROM AIRBRIDGE.PUBLIC.INTERNAL_MOBILE_EVENTS")
    print(f"  WHERE DATA__APP__APPID = {APP_ID}")
    print(f"  AND DATA__EVENTDATA__CATEGORY LIKE '9360%'")
    print(f"  GROUP BY 1 ORDER BY CNT DESC LIMIT 30;")
    df_events = None

---
## Step 3: 이벤트 매핑 확인

기본 커머스 매핑(`configs/default_commerce.json`)을 로드하고,  
이 앱의 실제 이벤트와 일치하는지 확인합니다.

- ✅ : 실제 데이터에 해당 이벤트가 존재
- ❌ 확인필요 : 데이터에 없거나 이벤트 이름이 다를 수 있음

모든 매핑이 맞으면 Step 5로 건너뛰어도 됩니다.  
틀린 게 있으면 Step 4에서 수정하세요.

In [ ]:
# 기본 커머스 매핑 로드
with open('../configs/default_commerce.json') as f:
    config = json.load(f)

config['app_id'] = APP_ID
config['app_name'] = APP_NAME

print("=== 기본 커머스 이벤트 매핑 ===")
for our_name, airbridge_name in config['event_mapping'].items():
    # 실제 데이터에 있는지 체크
    if df_events is not None and 'GOAL_CATEGORY' in df_events.columns:
        exists = airbridge_name in df_events['GOAL_CATEGORY'].values
    else:
        exists = None
    status = "✅" if exists == True else "❌ 확인필요"
    print(f"  {our_name:20s} → {airbridge_name:45s} {status}")

print()
print("매핑이 맞으면 다음 셀로 넘어가세요.")
print("틀린 게 있으면 아래에서 수정하세요:")

# === 이벤트 매핑 검증 ===
if df_events is not None:
    print()
    print("=== 이벤트 매핑 검증 ===")
    print()
    airbridge_events = set(df_events['GOAL_CATEGORY'].dropna().values)
    
    for our_name, expected_event in config['event_mapping'].items():
        if expected_event in airbridge_events:
            cnt = df_events[df_events['GOAL_CATEGORY'] == expected_event]['CNT'].values[0]
            print(f"  ✅ {our_name:20s} → {expected_event:45s} ({cnt:,}건)")
        else:
            print(f"  ❌ {our_name:20s} → {expected_event:45s} (이벤트 없음!)")
            print(f"     → 이 앱에서 실제로 쓰는 이벤트명을 확인하고 Step 4에서 수정하세요")
            # 유사한 이벤트 제안
            similar = [e for e in airbridge_events if our_name.replace('_', '') in str(e).lower().replace('.', '').replace('_', '')]
            if similar:
                print(f"     → 혹시 이건가요? {similar}")
    
    print()
    print("❌가 있으면 Step 4에서 매핑을 수정하세요. 전부 ✅면 Step 5로 넘어가세요.")
else:
    print()
    print("이벤트 데이터가 없어서 검증을 건너뜁니다. Step 2를 먼저 완료해주세요.")

---
## Step 4: 매핑 수정 (필요시)

Step 3에서 ❌ 표시된 이벤트가 있으면 여기서 수정합니다.  
앱마다 이벤트 이름이 다를 수 있으니, Airbridge 대시보드 또는 Step 2 결과를 보고 맞는 이벤트명으로 바꿔주세요.

수정이 필요 없으면 이 셀은 그대로 실행만 하면 됩니다.

In [ ]:
# 매핑 수정이 필요하면 여기서 수정
# 예: config['event_mapping']['purchase'] = 'order_complete'

# 확인
print("최종 매핑:")
for k, v in config['event_mapping'].items():
    print(f"  {k}: {v}")

---
## Step 5: config 저장

확정된 이벤트 매핑을 `configs/{APP_NAME}.json`으로 저장합니다.  
이 파일은 데이터 파이프라인과 서버에서 참조합니다.

In [ ]:
import os
os.makedirs('../configs', exist_ok=True)
config_path = f'../configs/{APP_NAME}.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print(f"✅ {config_path} 저장 완료!")

---
## Step 6: 데이터 추출 테스트

매핑이 맞는지 소량 데이터로 테스트합니다.  
실서비스에서는 Snowflake에서 최근 3일 데이터를 추출하여 feature 생성이 정상적인지 확인합니다.

프로토타입에서는 기존 샘플 데이터로 모델을 학습합니다.

In [ ]:
# 실서비스: Snowflake에서 소량 추출
# df_test = pd.read_sql("""
#     SELECT * FROM AIRBRIDGE.PUBLIC.INTERNAL_MOBILE_EVENTS
#     WHERE DATA__APP__APPID = {APP_ID}
#     AND DATA__TIMESTAMP >= DATEADD('day', -3, CURRENT_DATE())
#     LIMIT 1000
# """, conn)
# print(f"추출 건수: {len(df_test)}")
# print(df_test['DATA__EVENTDATA__GOAL__CATEGORY'].value_counts().head(10))

# 프로토타입: 기존 sample 데이터 사용
print("데이터 추출 테스트는 실서비스 Snowflake 연동 후 가능합니다.")
print("프로토타입에서는 기존 sample 데이터로 모델을 학습합니다.")

---
## Step 6.5: 온보딩 학습 데이터 준비

Snowflake에서 과거 30~60일 데이터를 추출합니다.  
아래 셀을 실행하면 정확한 Snowflake 쿼리가 출력됩니다.

**온보딩 데이터는 주간 학습 데이터와 다릅니다:**
- `assigned_trigger`, `is_random`, `modal_clicked` 컬럼이 **필요 없습니다** (아직 RCT 전이니까)
- 기간을 넉넉하게 잡을수록 D3 모델이 정확해집니다

In [ ]:
from datetime import datetime, timedelta

today = datetime.now().strftime('%Y-%m-%d')
onboard_start = (datetime.now() - timedelta(days=60)).strftime('%Y-%m-%d')
onboard_end = (datetime.now() - timedelta(days=3)).strftime('%Y-%m-%d')

# config에서 이벤트 매핑 가져오기
event_mapping = config.get('event_mapping', {})
ev_product_viewed = event_mapping.get('product_viewed', 'airbridge.ecommerce.product.viewed')
ev_signin = event_mapping.get('signin', 'airbridge.user.signin')
ev_addedtocart = event_mapping.get('addedtocart', 'airbridge.ecommerce.product.addedToCart')
ev_home_viewed = event_mapping.get('home_viewed', 'airbridge.ecommerce.home.viewed')
ev_addtowishlist = event_mapping.get('addtowishlist', 'airbridge.addToWishlist')
ev_onboarding = event_mapping.get('onboarding', 'airbridge.onboarding')
ev_signup = event_mapping.get('signup', 'airbridge.user.signup')
ev_purchase = event_mapping.get('purchase', 'airbridge.ecommerce.order.completed')
ev_deeplink_cat = event_mapping.get('deeplink_open_category', '9162')
leakage_events = config.get('leakage_events', [ev_purchase, 'airbridge.initiateCheckout'])

data_dir = f'../data/{APP_NAME}'
os.makedirs(data_dir, exist_ok=True)
save_path = f'{data_dir}/weekly_{today}.csv'

print(f"""
{'='*70}
온보딩 학습 데이터를 준비하세요.

Snowflake에서 아래 쿼리를 실행하고 CSV로 저장:
파일명: {save_path}
기간: {onboard_start} ~ {onboard_end} (과거 60일, D3 outcome 확정분)
{'='*70}

-- ============================================================
-- Step 1: install_users (설치 유저 추출)
-- ============================================================
CREATE TEMPORARY TABLE install_users AS
SELECT
    DATA__DEVICE__AIRBRIDGEGENERATEDDEVICEUUID AS user_id,
    MIN(EVENT_TIMESTAMP) AS install_ts,
    MAX(DATA__DEVICE__OSNAME) AS OS_NAME,
    MAX(DATA__DEVICE__MANUFACTURER) AS DEVICE_MANUFACTURER,
    MAX(DATA__DEVICE__LANGUAGE) AS DEVICE_LANGUAGE,
    MAX(DATA__DEVICE__TIMEZONE) AS DEVICE_TIMEZONE,
    MAX(DATA__DEVICE__OSVERSION) AS DEVICE_OSVERSION,
    MAX(DATA__DEVICE__NETWORK__CARRIER) AS DEVICE_CARRIER
FROM AIRBRIDGE.PUBLIC.INTERNAL_MOBILE_EVENTS
WHERE DATA__APP__APPID = {APP_ID}
  AND DATA__EVENTDATA__CATEGORY = '9161'
  AND EVENT_TIMESTAMP >= '{onboard_start}'
  AND EVENT_TIMESTAMP < '{onboard_end}'
GROUP BY 1;

-- ============================================================
-- Step 2: fmt_flat (UA attribution 결과)
-- ============================================================
CREATE TEMPORARY TABLE fmt_flat AS
SELECT
    DATA__DEVICE:airbridgeGeneratedDeviceUUID::STRING AS user_id,
    DATA__ATTRIBUTIONRESULT AS journey_json,
    EVENT_TIMESTAMP AS fmt_ts
FROM AIRBRIDGE.PUBLIC.AIRBRIDGE_FMT_MOBILE_APP_EVENT_RESULTS_V20210802
WHERE APP_ID = {APP_ID}
  AND DATA__EVENTDATA__CATEGORY = '9161'
QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY fmt_ts DESC) = 1;

-- ============================================================
-- Step 3: ua_table (유저 + journey JOIN)
-- ============================================================
CREATE TEMPORARY TABLE ua_table AS
SELECT iu.*, ff.journey_json
FROM install_users iu
LEFT JOIN fmt_flat ff ON iu.user_id = ff.user_id;

-- ============================================================
-- Step 4: inapp_base (인앱 이벤트)
-- ============================================================
CREATE TEMPORARY TABLE inapp_base AS
SELECT
    ev.DATA__DEVICE__AIRBRIDGEGENERATEDDEVICEUUID AS user_id,
    ev.DATA__EVENTDATA__CATEGORY AS evt_cat,
    ev.DATA__EVENTDATA__GOAL__CATEGORY AS goal_cat,
    ev.EVENT_TIMESTAMP,
    TIMESTAMPDIFF(SECOND, iu.install_ts, ev.EVENT_TIMESTAMP) AS sec_after
FROM AIRBRIDGE.PUBLIC.INTERNAL_MOBILE_EVENTS ev
INNER JOIN install_users iu
    ON ev.DATA__DEVICE__AIRBRIDGEGENERATEDDEVICEUUID = iu.user_id
WHERE ev.DATA__APP__APPID = {APP_ID}
  AND ev.EVENT_TIMESTAMP >= iu.install_ts
  AND ev.EVENT_TIMESTAMP <= DATEADD(DAY, 30, iu.install_ts);

-- ============================================================
-- Step 5: user_churn_base (이탈 기준)
-- ============================================================
CREATE TEMPORARY TABLE user_churn_base AS
SELECT user_id, MAX(sec_after) AS seconds_active_after_install
FROM inapp_base GROUP BY 1;

-- ============================================================
-- Step 6: inapp_agg_5min (인앱 5분 집계 + d3 outcomes)
-- ============================================================
CREATE TEMPORARY TABLE inapp_agg_5min AS
SELECT
    user_id,
    COUNT(CASE WHEN goal_cat = '{ev_product_viewed}' AND sec_after <= 300 THEN 1 END) AS product_viewed_count,
    MAX(CASE WHEN goal_cat = '{ev_signin}' AND sec_after <= 300 THEN 1 ELSE 0 END) AS user_signin,
    MAX(CASE WHEN goal_cat = '{ev_addedtocart}' AND sec_after <= 300 THEN 1 ELSE 0 END) AS product_addedtocart,
    MAX(CASE WHEN evt_cat = '{ev_deeplink_cat}' AND sec_after <= 300 THEN 1 ELSE 0 END) AS deeplink_open,
    MAX(CASE WHEN goal_cat = '{ev_home_viewed}' AND sec_after <= 300 THEN 1 ELSE 0 END) AS home_viewed,
    MAX(CASE WHEN goal_cat = '{ev_addtowishlist}' AND sec_after <= 300 THEN 1 ELSE 0 END) AS addtowishlist,
    MAX(CASE WHEN goal_cat = '{ev_onboarding}' AND sec_after <= 300 THEN 1 ELSE 0 END) AS onboarding,
    MAX(CASE WHEN goal_cat = '{ev_signup}' AND sec_after <= 300 THEN 1 ELSE 0 END) AS user_signup,
    COUNT(CASE WHEN sec_after <= 300 AND evt_cat != '9161'
               AND COALESCE(goal_cat, '') NOT IN ('{leakage_events[0]}', '{leakage_events[1] if len(leakage_events) > 1 else ""}')
               THEN 1 END) AS total_events,
    COUNT(DISTINCT CASE WHEN sec_after <= 300 AND evt_cat != '9161'
               AND COALESCE(goal_cat, '') NOT IN ('{leakage_events[0]}', '{leakage_events[1] if len(leakage_events) > 1 else ""}')
               THEN COALESCE(goal_cat, evt_cat) END) AS n_event_types,
    MAX(CASE WHEN goal_cat = '{ev_purchase}' AND sec_after <= 259200 THEN 1 ELSE 0 END) AS d3_purchase
FROM inapp_base GROUP BY 1;

-- ============================================================
-- Facebook 터치포인트 복원 (선택)
-- ============================================================
CREATE TEMPORARY TABLE ua_table_enriched AS
SELECT ut.*, rf.meta_touchpoint
FROM ua_table ut
LEFT JOIN (
    SELECT ru.user_id, fb.DATA AS meta_touchpoint
    FROM ua_table ru
    INNER JOIN AB180_X_FACEBOOK.PUBLIC.AIRBRIDGE_PLATFORM_MATCHED_TOUCHPOINTS_V20230410 fb
        ON fb.APP_ID = {APP_ID}
        AND fb.CONVERSION_EVENT_TIMESTAMP = ru.install_ts
    WHERE ru.journey_json LIKE '%"restricted"%'
) rf ON ut.user_id = rf.user_id;

-- ============================================================
-- 최종 결합 쿼리 (이 결과를 CSV로 다운로드)
-- ============================================================
SELECT
    {APP_ID} AS app_id,
    ue.user_id AS airbridge_uuid,
    ue.install_ts,
    ue.OS_NAME, ue.DEVICE_MANUFACTURER, ue.DEVICE_LANGUAGE,
    ue.DEVICE_TIMEZONE, ue.DEVICE_OSVERSION, ue.DEVICE_CARRIER,
    ue.journey_json, ue.meta_touchpoint,
    COALESCE(ia.product_viewed_count, 0) AS product_viewed_count,
    COALESCE(ia.user_signin, 0) AS user_signin,
    COALESCE(ia.product_addedtocart, 0) AS product_addedtocart,
    COALESCE(ia.deeplink_open, 0) AS deeplink_open,
    COALESCE(ia.home_viewed, 0) AS home_viewed,
    COALESCE(ia.addtowishlist, 0) AS addtowishlist,
    COALESCE(ia.onboarding, 0) AS onboarding,
    COALESCE(ia.user_signup, 0) AS user_signup,
    COALESCE(ia.total_events, 0) AS total_events,
    COALESCE(ia.n_event_types, 0) AS n_event_types,
    COALESCE(ia.d3_purchase, 0) AS d3_purchase,
    CASE WHEN COALESCE(uc.seconds_active_after_install, 0) <= 259200 THEN 1 ELSE 0 END AS d3_churn,
    COALESCE(ta.IS_HAS_FRAUD, 0) AS IS_HAS_FRAUD,
    ta.sa_term_list
FROM ua_table_enriched ue
LEFT JOIN inapp_agg_5min ia ON ue.user_id = ia.user_id
LEFT JOIN user_churn_base uc ON ue.user_id = uc.user_id
LEFT JOIN touchpoint_analysis ta ON ue.user_id = ta.user_id;

-- ============================================================
-- NOTE: assigned_trigger, is_random, modal_clicked 컬럼은
-- 온보딩 데이터에는 필요 없습니다. (RCT 전이니까)
-- 이 컬럼들은 weekly_training.ipynb에서 Supabase JOIN으로 추가됩니다.
-- ============================================================
""")

print(f"CSV를 {save_path} 에 저장한 후 다음 셀(Step 7)을 실행하세요.")

---
## Step 7: D3 Purchase/Churn 모델 학습

25개 feature(UA 15개 + In-app 10개)로 두 가지 모델을 학습합니다:

| 모델 | 예측 대상 | 용도 |
|------|----------|------|
| D3 Purchase | 3일 내 구매 확률 | 고가치 유저 식별 |
| D3 Churn | 3일 내 이탈 확률 | 이탈 위험 유저 식별 |

### 온보딩 vs 매주 학습의 차이

| | 온보딩 (지금) | 매주 학습 |
|---|---|---|
| **데이터 범위** | **과거 30~60일** (넉넉하게) | 최근 1주일 |
| **RCT 태그 필요?** | ❌ 필요 없음 | ✅ Supabase JOIN 필요 |
| **CATE 학습?** | ❌ 못 함 (RCT 데이터 없음) | ✅ is_random=1 유저로 학습 |
| **Supabase JOIN?** | ❌ 필요 없음 | ✅ assigned_trigger, is_random 가져옴 |

**온보딩 때는 과거 데이터를 많이 뽑을수록 D3 모델이 정확해집니다.**
RCT와 무관하게 "이 유저가 구매할지/이탈할지"만 예측하는 모델이니까요.

### 데이터 준비 방법

Snowflake에서 아래 기간으로 데이터를 추출하세요:
```
온보딩: start_date = 오늘 - 60일, end_date = 오늘 - 3일
매주:   start_date = 오늘 - 10일, end_date = 오늘 - 3일
```

추출한 CSV를 `data/{APP_NAME}/weekly_YYYY-MM-DD.csv`로 저장하면 아래 코드가 자동으로 로드합니다.
SQL과 CSV 포맷은 `docs/weekly_data_format.md`를 참고하세요.

**⚠️ 온보딩 데이터에는 `assigned_trigger`, `is_random`, `modal_clicked` 컬럼이 없어도 됩니다.**
이 컬럼들은 CATE 학습에만 쓰이고, D3 Purchase/Churn 학습에는 필요하지 않습니다.

In [ ]:
import numpy as np
import glob
import pickle
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# Feature definitions (must match server/predict.py)
UA_FEATURES = [
    'trackinglink_count', 'DA_count', 'SA_count',
    'unique_channel_count', 'channel_entropy', 'last_touch_is_da',
    'latency', 'recency', 'recent_touch_pressure',
    'touch_per_latency_hour', 'last1h_touch_count', 'recent_24h_ratio',
    'click_ratio', 'impression_count', 'is_single_touch_install'
]

INAPP_FEATURES = [
    'product_viewed_count', 'user_signin', 'product_addedtocart',
    'deeplink_open', 'home_viewed', 'addtowishlist',
    'onboarding', 'user_signup', 'total_events', 'n_event_types'
]

ALL_FEATURES = UA_FEATURES + INAPP_FEATURES
print(f"Feature 수: UA {len(UA_FEATURES)}개 + In-app {len(INAPP_FEATURES)}개 = 전체 {len(ALL_FEATURES)}개")

# Model output directory
model_dir = f'../models/{APP_NAME}'
os.makedirs(model_dir, exist_ok=True)

# 앱별 데이터 폴더
data_dir = f'../data/{APP_NAME}'
os.makedirs(data_dir, exist_ok=True)

# 최신 weekly 데이터 자동 로드
weekly_files = sorted(glob.glob(f'{data_dir}/weekly_*.csv'))
if weekly_files:
    latest_file = weekly_files[-1]  # 날짜순 정렬 → 마지막이 최신
    df = pd.read_csv(latest_file)
    print(f"✅ {latest_file} 로드 ({len(df)}명)")
    print(f"   파일 {len(weekly_files)}개 중 최신 사용")
else:
    print(f"⚠️ {data_dir}/weekly_*.csv 파일이 없습니다!")
    print(f"")
    print(f"Snowflake에서 데이터를 추출하여 아래 경로에 저장해주세요:")
    print(f"  {data_dir}/weekly_{datetime.now().strftime('%Y-%m-%d')}.csv")
    print(f"")
    print(f"CSV 포맷은 docs/weekly_data_format.md를 참고하세요.")
    df = None

# 온보딩 시 초기 데이터 저장 (있는 경우)
# save_path = f'../data/{APP_NAME}/weekly_{datetime.now().strftime("%Y-%m-%d")}.csv'

if df is not None:
    print(f"D3 구매율: {df['d3_purchase'].mean():.1%}")
    print(f"D3 이탈율: {df['d3_churn'].mean():.1%}")

    X_all = df[ALL_FEATURES].values

    # --- D3 Purchase Model ---
    print("\n=== D3 Purchase 모델 학습 ===")
    y_purchase = df['d3_purchase'].values
    purchase_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
    purchase_model.fit(X_all, y_purchase)

    scores_purchase = cross_val_score(purchase_model, X_all, y_purchase, cv=5, scoring='roc_auc')
    print(f"D3 Purchase AUC: {scores_purchase.mean():.3f} (±{scores_purchase.std():.3f})")

    purchase_data = {'model': purchase_model, 'feature_names': ALL_FEATURES}
    with open(f'{model_dir}/d3_purchase_model.pkl', 'wb') as f:
        pickle.dump(purchase_data, f)
    print(f"{model_dir}/d3_purchase_model.pkl 저장 완료!")

    # --- D3 Churn Model ---
    print("\n=== D3 Churn 모델 학습 ===")
    y_churn = df['d3_churn'].values
    churn_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
    churn_model.fit(X_all, y_churn)

    scores_churn = cross_val_score(churn_model, X_all, y_churn, cv=5, scoring='roc_auc')
    print(f"D3 Churn AUC: {scores_churn.mean():.3f} (±{scores_churn.std():.3f})")

    churn_data = {'model': churn_model, 'feature_names': ALL_FEATURES}
    with open(f'{model_dir}/d3_churn_model.pkl', 'wb') as f:
        pickle.dump(churn_data, f)
    print(f"{model_dir}/d3_churn_model.pkl 저장 완료!")

    # --- Summary ---
    print("\n=== 학습 결과 요약 ===")
    print(f"  D3 Purchase AUC: {scores_purchase.mean():.3f}  (0.5=동전던지기, 0.8+=실무에서 쓸만함)")
    print(f"  D3 Churn AUC:    {scores_churn.mean():.3f}")
    print(f"  학습 데이터:      {len(df)}명 (전체 유저)")
    print(f"  저장 위치:        {model_dir}/")

---
## Step 8: 서버에 업로드

학습된 모델 파일을 서버에 업로드합니다.  
서버가 자동으로 모델을 로드하므로 재배포 없이 바로 반영됩니다.

CATE 모델이 아직 없기 때문에 **exploration 모드**로 시작합니다:
- 모든 유저에게 trigger를 랜덤 배정
- D3 Purchase/Churn 확률은 바로 제공
- CATE 모델은 RCT 데이터가 충분히 쌓인 후 `weekly_training.ipynb`에서 학습

In [ ]:
import requests

model_files = [
    f"../models/{APP_NAME}/d3_purchase_model.pkl",
    f"../models/{APP_NAME}/d3_churn_model.pkl",
]

print(f"=== 서버에 모델 업로드 ({APP_NAME}) ===")
for filepath in model_files:
    filename = os.path.basename(filepath)
    if not os.path.exists(filepath):
        print(f"  [SKIP] {filename}")
        continue
    try:
        with open(filepath, "rb") as f:
            resp = requests.post(f"{SERVER_URL}/v1/models/{APP_NAME}/upload", files={"file": (filename, f)})
        if resp.status_code == 200:
            print(f"  [OK] {filename} → {resp.json()['mode']}")
        else:
            print(f"  [ERROR] {resp.status_code}: {resp.text}")
    except requests.ConnectionError:
        print(f"  [ERROR] 서버 연결 실패: {SERVER_URL}")
        break

print()
print("✅ 서버가 exploration 모드로 시작됩니다.")
print("   → 모든 유저에게 trigger를 랜덤 배정")
print("   → D3 Purchase/Churn 확률은 바로 제공")
print("   → CATE 모델은 RCT 데이터 충분히 쌓인 후 weekly_training에서 학습")

---
## Step 8.5: Feature Store 유저 추가

⚠️ **중요**: 모델을 서버에 올렸더라도, Feature Store에 이 앱의 유저가 없으면 API가 404를 반환합니다.

**프로토타입**: `data/feature_store.csv`에 유저를 수동 추가
**실서비스**: Airbridge DB와 실시간 연동 (엔지니어링팀과 협의 필요)

In [ ]:
# Feature Store에 이 앱의 유저가 있는지 확인
import requests
try:
    resp = requests.get(f"{SERVER_URL}/v1/users/{APP_NAME}")
    if resp.status_code == 200:
        data = resp.json()
        print(f"✅ Feature Store에 {APP_NAME} 유저 {data['total']}명 존재")
    else:
        print(f"⚠️ Feature Store에 {APP_NAME} 유저가 없습니다!")
        print(f"")
        print(f"data/feature_store.csv에 유저를 추가해야 합니다.")
        print(f"필요한 컬럼: app_id, airbridge_uuid, {', '.join(ALL_FEATURES)}")
        print(f"")
        print(f"프로토타입에서는 data/generate_feature_store.py를 참고하여 샘플 데이터를 생성할 수 있습니다.")
except:
    print("서버에 연결할 수 없습니다.")

---
## Step 9: 고객사 안내사항

고객사에 전달해야 할 기술 연동 가이드입니다.  
아래 출력 내용을 복사하여 고객사 담당자에게 전달하세요.

포함 내용:
1. Airbridge SDK 이벤트 태깅 확인사항
2. `entry_modal_clicked` 이벤트 태깅 코드 (iOS/Android)
3. 모달 4종 디자인 가이드
4. API 연동 엔드포인트
5. 서비스 타임라인 (exploration → optimized)

In [ ]:
print(f"""
{'='*60}
고객사 안내사항 ({APP_NAME})
{'='*60}

1. Airbridge SDK 이벤트 태깅
   - 기존 이벤트 (product.viewed, signin 등)가 태깅되어 있는지 확인
   - 추가 태깅 필요: entry_modal_clicked

2. entry_modal_clicked 이벤트 태깅 코드:

   [iOS - Swift]
   Airbridge.trackEvent(
       category: "entry_modal_clicked",
       semanticAttributes: [:],
       customAttributes: [
           "trigger_type": "social_proof"  // price_appeal | social_proof | scarcity | novelty
       ]
   )

   [Android - Kotlin]
   Airbridge.trackEvent(
       "entry_modal_clicked",
       mapOf(),
       mapOf("trigger_type" to "social_proof")
   )

   ⚠️ Airbridge 대시보드에서 Event Taxonomy에 'entry_modal_clicked' 등록 필수!

3. 모달 4개 디자인 (포맷 동일하게):
   - Price Appeal: 할인/쿠폰 강조
   - Social Proof: 인기/랭킹 강조
   - Scarcity: 한정/긴급성 강조
   - Novelty: 신상품/트렌드 강조

4. API 연동:
   POST {SERVER_URL}/v1/entry/predict
   Body: {{"app_id": "{APP_NAME}", "airbridge_uuid": "<유저UUID>"}}

5. 타임라인:
   - 지금: exploration 모드 (trigger 랜덤 배정 + D3 예측 제공)
   - 2-4주 후: 충분한 데이터 수집 → CATE 모델 학습 → optimized 모드 전환
""")